# tofu_v10 — session C (self-contained)

The three final referee experiments: question-boundary decoding (no answer conditioning), late-layer activation patching (the causal test), and the in-context channel on the oracle checkpoints. Run top to bottom; ~2 h total. Everything resumable as before.


### Cell 1 — mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 2 — packages


In [ ]:
%pip install -q -U transformers peft accelerate datasets bitsandbytes sentencepiece rouge-score

### Cell 3 — unpack all scripts (embedded; nothing to drag)


In [ ]:
import base64, zlib, json, os
BLOB = (
    "..."
)
scripts = json.loads(zlib.decompress(base64.b64decode("".join(BLOB))))
os.chdir("/content")
for name, text in scripts.items():
    open(name, "w").write(text)
    print("wrote", name)

### Cell 4 — restore adapters (incl. the v9 oracle and recovered checkpoints)


In [ ]:
import os
need = ([f"s{i}/retain90" for i in range(10)] +
        [f"s{i}/{d}" for i in "012" for d in ["full", "unlearn_neggrad", "oraclex", "benignrec9"]])
for d in need:
    os.makedirs(f"/content/adapters/{d.split('/')[0]}", exist_ok=True)
    !cp -r "/content/drive/MyDrive/tofu_v3_backup/adapters/{d}" "/content/adapters/{d}" 2>/dev/null
!ls /content/adapters/s0/

### Cell 5 — load the library


In [ ]:
import sys, os
os.chdir("/content")
for f in ["tofu_all.py", "tofu_v3.py", "tofu_v4.py", "tofu_v5.py",
          "tofu_v6.py", "tofu_v7.py", "tofu_v8.py", "tofu_v9.py"]:
    print("loading", f, flush=True)
    sys.argv = [f]
    exec(open(f).read())
print("library loaded")

### Cell 6 — define backup()


In [ ]:
import subprocess
def backup():
    subprocess.run(["bash", "-c",
      "mkdir -p '/content/drive/MyDrive/tofu_v3_backup/results_v10' && "
      "cp -r results_v10/. '/content/drive/MyDrive/tofu_v3_backup/results_v10/' 2>/dev/null; true"])
    print("backed up results_v10 to Drive")
backup

### Cell 7 — driver (~1.5-2 h)


In [ ]:
import sys
STAGES = ([("lens2", s) for s in "012"] + [("patch", s) for s in "012"] +
          [("oracleicl", s) for s in "012"])
for stage, seed in STAGES:
    print(f"\n===== v10 {stage} {seed} =====", flush=True)
    sys.argv = ["tofu_v10.py", stage, seed]
    exec(open("tofu_v10.py").read())
    backup()
sys.argv = ["tofu_v10.py", "figures10"]; exec(open("tofu_v10.py").read())
print("V10 COMPLETE")

### Cell 8 — zip to Drive


In [ ]:
!cd /content && zip -qr results_v10.zip results_v10
!cp /content/results_v10.zip "/content/drive/MyDrive/"
print("results_v10.zip is in MyDrive — download it into the Persistence of Memory folder")